In [2]:
import torch
from yolov5.models.common import DetectMultiBackend
from yolov5.utils.dataloaders import LoadImages
from yolov5.utils.general import non_max_suppression, scale_boxes, xyxy2xywh
from yolov5.utils.plots import Annotator
import cv2

class Train2():
    def __init__(self):
        # 加載 YOLOv5 模型
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = DetectMultiBackend('./model/weights.pt', device=self.device)  # 替換成你的 YOLOv5 權重路徑
        self.stride, self.names, self.pt = self.model.stride, self.model.names, self.model.pt

    # 測試圖片路徑
    
    def load_image(self, image_path):
    # 加載圖片
        dataloader = LoadImages(image_path, img_size=640, stride=self.stride, auto=self.pt)
        for path, img, im0s, vid_cap, s in dataloader:
            img = torch.from_numpy(img).to(self.device)
            img = img.half() if self.model.fp16 else img.float()  # 如果支持 FP16，轉為半精度
            img /= 255.0  # 歸一化到 [0, 1]
            if len(img.shape) == 3:
                img = img[None]  # 添加一個 batch 維度

            # 推理
            pred = self.model(img, augment=False, visualize=False)

            # 非極大值抑制（NMS）
            pred = non_max_suppression(pred, conf_thres=0.5, iou_thres=0.45, classes=None, agnostic=False)

            # 繪製檢測結果
            for i, det in enumerate(pred):  # 每張圖片的檢測結果
                im0 = im0s.copy()

                annotator = Annotator(im0, line_width=2, example=str(self.names))
                if len(det):
                    # 將邊界框大小調整回原始圖片
                    det[:, :4] = scale_boxes(img.shape[2:], det[:, :4], im0.shape).round()

                    # 遍歷每個檢測到的物件
                    for *xyxy, conf, cls in reversed(det):
                        label = f"{self.names[int(cls)]} {conf:.2f}"  # 類別名稱和置信度
                        print(label)

                        # 繪製邊界框和標籤
                        annotator.box_label(xyxy, label, color=(0, 255, 0))

                # 顯示圖片
                im0 = annotator.result()
        return im0


if __name__ == "__main__":
    image_path = "./dataset/Original img/image copy 4.png"
    yolo = Train2()
    im0 = yolo.load_image(image_path)

    cv2.imshow("Detection Results", im0)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

c:\Users\corps\anaconda3\envs\capoo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fusing layers... 
YOLOv5n6 summary: 281 layers, 3107836 parameters, 0 gradients, 4.3 GFLOPs


Ground 0.50
Ground 0.79
Player 0.82
Player 0.85
Ground 0.85
Snail 0.85
Snail 0.88
Slime 0.89
Player 0.90
Ladder 0.91
Player 0.95


In [ ]:
import torch
import cv2
from yolov5.models.common import DetectMultiBackend
from yolov5.utils.general import non_max_suppression, scale_boxes
from yolov5.utils.plots import Annotator
from yolov5.utils.augmentations import letterbox
import numpy as np

class Main():
    def __init__(self):
        # 加載 YOLOv5 模型
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = DetectMultiBackend('./model/weights.pt', device=self.device)  # 替換成你的 YOLOv5 權重路徑
        self.stride, self.names, self.pt = self.model.stride, self.model.names, self.model.pt

    def load_image(self, img):
        im0 = img.copy()

        # letterbox處理 (縮放、填充使得圖片符合 YOLO 模型輸入要求)
        img, ratio, dwdh = letterbox(img, 640, stride=self.stride, auto=self.pt)
        # img 此時為 (H, W, C) 格式，且是 RGB 
        # 若 letterbox 函數保留原本順序為 BGR，要自行轉 RGB；若無特別註明，此處皆為 BGR，因此需轉成 RGB
        # 但 YOLOv5 模型預設以 RGB 為輸入，所以要先將 im0(BGR) -> RGB
        # 若要保留顯示 im0 在後面標註時維持 BGR 可，只需要在推論前轉 img 為 RGB 即可
        # 下面程式碼假設 letterbox 只是縮放，未改變通道順序，所以我們先轉 RGB:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 轉成張量並調整維度： (H, W, C) -> (C, H, W)
        img = np.ascontiguousarray(img)
        img = torch.from_numpy(img).to(self.device)
        img = img.permute(2, 0, 1)  # (H, W, C) -> (C, H, W)
        img = img.half() if self.model.fp16 else img.float()  # FP16 或 FP32
        img /= 255.0
        if img.ndimension() == 3:
            img = img.unsqueeze(0)  # [C, H, W] -> [1, C, H, W]

        # 推理
        pred = self.model(img, augment=False, visualize=False)

        # 非極大值抑制（NMS）
        pred = non_max_suppression(pred, conf_thres=0.5, iou_thres=0.45, classes=None, agnostic=False)

        # 繪製檢測結果
        for i, det in enumerate(pred):
            annotator = Annotator(im0, line_width=2, example=str(self.names))
            if len(det):
                # 將邊界框從縮放後大小還原到原圖大小
                det[:, :4] = scale_boxes(img.shape[2:], det[:, :4], im0.shape).round()

                for *xyxy, conf, cls in reversed(det):
                    label = f"{self.names[int(cls)]} {conf:.2f}"
                    print(label)
                    annotator.box_label(xyxy, label, color=(0, 255, 0))

            im0 = annotator.result()

        return im0

if __name__ == "__main__":
    image_path = "./dataset/Original img/image copy 4.png"
    img = cv2.imread(image_path)  # BGR 格式
    yolo = Main()
    im0 = yolo.load_image(img)

    cv2.imshow("Detection Results", im0)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


Fusing layers... 
YOLOv5n6 summary: 281 layers, 3107836 parameters, 0 gradients, 4.3 GFLOPs


Ground 0.50
Ground 0.79
Player 0.82
Player 0.85
Ground 0.85
Snail 0.85
Snail 0.88
Slime 0.89
Player 0.90
Ladder 0.91
Player 0.95


In [10]:
from yolov5.models.common import DetectMultiBackend
import torch

# 載入模型
model_path = "./model/weights.pt"  # 替換為你的權重路徑
model = DetectMultiBackend(model_path)

# 提取類別名稱與數量
if hasattr(model, 'names'):
    class_names = model.names
    print(f"總共有 {len(class_names)} 個類別")
    print("類別名稱:", class_names)

# 提取訓練相關信息
if hasattr(model, 'trainer') and model.trainer:
    trainer_info = model.trainer.opt
    print("訓練相關資訊:")
    print(f"  Epoch 數量: {trainer_info.get('epochs', '未知')}")
    print(f"  Batch size: {trainer_info.get('batch_size', '未知')}")
    print(f"  初始學習率: {trainer_info.get('lr0', '未知')}")
else:
    print("訓練相關資訊不可用（可能未保存於模型中）。")

# 顯示模型架構
print("\n模型架構:")
print(model.model)


Fusing layers... 
YOLOv5n6 summary: 281 layers, 3107836 parameters, 0 gradients, 4.3 GFLOPs


總共有 8 個類別
類別名稱: {0: 'Ground', 1: 'Ladder', 2: 'Mushroom', 3: 'Pig', 4: 'Player', 5: 'Slime', 6: 'Snail', 7: 'Stump'}
訓練相關資訊不可用（可能未保存於模型中）。

模型架構:
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3(
      (cv1): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=Fal